In [0]:
# Databricks notebook source

#  - Semi-Structured Data and VARIANT
 
# 1. Imports

from pyspark.sql.functions import col, parse_json
 



In [0]:


%sql
--# 2. Add VARIANT column to transactions table
ALTER TABLE retail.sales.transactions
ADD COLUMN metadata VARIANT;

DESCRIBE TABLE retail.sales.transactions;

In [0]:
# 3. Create JSON metadata staging data

metadata_rows = [
    ("TXN-00001", '{"channel": "online", "device": "mobile", "promo_applied": true}'),
    ("TXN-00002", '{"channel": "physical", "device": null, "promo_applied": false}'),
    ("TXN-00003", '{"channel": "online", "device": "desktop", "promo_applied": true}'),
    ("TXN-00004", '{"channel": "online", "device": "mobile", "promo_applied": false}'),
    ("TXN-00005", '{"channel": "physical", "device": null, "promo_applied": false}')
]

df_meta = spark.createDataFrame(
    metadata_rows,
    ["transaction_id", "metadata_json"]
)

df_meta_variant = (
    df_meta
    .withColumn("metadata", parse_json(col("metadata_json")))
    .drop("metadata_json")
)

display(df_meta_variant)

In [0]:
df_meta_variant.createOrReplaceTempView("metadata_updates")

print("metadata_updates registered")

In [0]:

%sql
--# 4. Check transactions before updating metadata


SELECT
    transaction_id,
    total_amount
FROM retail.sales.transactions
WHERE transaction_id IN (
    'TXN-00001',
    'TXN-00002',
    'TXN-00003',
    'TXN-00004',
    'TXN-00005'
);


In [0]:



%sql
--# 5. Merge metadata into transactions

MERGE INTO retail.sales.transactions AS target
USING metadata_updates AS source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
    UPDATE SET target.metadata = source.metadata;


In [0]:
%sql
--# 7. Extract fields from VARIANT
SELECT
    transaction_id,
    total_amount,
    metadata:channel::string AS channel,
    metadata:device::string AS device,
    metadata:promo_applied::boolean AS promo_applied
FROM retail.sales.transactions
WHERE metadata IS NOT NULL;


In [0]:
%sql
--# 8. Filter using a nested VARIANT field

SELECT
    transaction_id,
    transaction_date,
    total_amount,
    metadata:channel::string AS channel
FROM retail.sales.transactions
WHERE metadata:channel::string = 'online';